# Fase 1 · Corpus de transcripciones oficiales del BCE (Bloque 2)

**Objetivo:** construir una tabla, frase a frase, con todas las ruedas de prensa de la era Lagarde (diciembre de 2019 – septiembre de 2026): declaración inicial, preguntas de los periodistas y respuestas del panel. Es la base del clasificador de postura (fase 2), del histórico (fase 3) y del chat (fase 4).

## 1. Fuente de datos

Usamos las páginas HTML oficiales del BCE en lugar del dataset de discursos que publica el propio banco. El dataset es texto plano, mientras que el HTML conserva la estructura que necesitamos: las preguntas de los periodistas aparecen en negrita y, cuando responden presidenta y vicepresidente, cada intervención empieza por una etiqueta con su nombre. Eso permite separar declaración, preguntas y respuestas, y atribuir cada respuesta a su hablante, sin heurísticas sobre el contenido. El BCE permite reproducir sus contenidos citando la fuente.

La página índice del BCE no incluye el listado de ruedas de prensa en su HTML, así que lo leemos de los ficheros que la web carga para cada año.

In [1]:
# autoreload aplica los cambios de src/ sin reiniciar el kernel
%load_ext autoreload
%autoreload 2

import time
import pandas as pd

from src.config import HISTORY_DIR, RAW_DIR
from src.text.corpus import listar_urls, descargar_html, construir_corpus, resumen_por_rueda

## 2. Listado de ruedas de prensa

Recorremos los años 2019-2026 y nos quedamos con las ruedas de Lagarde (la primera es del 12 de diciembre de 2019). Control esperado: 8 ruedas por año, 1 en 2019 y 6 en 2026 (hasta septiembre).

In [2]:
# Recorremos los años de la era Lagarde (2019 incluye todavía ruedas de Draghi)
urls = {}
for year in range(2019, 2027):
    urls.update(listar_urls(year))
    time.sleep(1)  # pausa de cortesía entre peticiones a la web del BCE

# Solo las ruedas de Lagarde
urls = dict(sorted((f, u) for f, u in urls.items() if f >= "2019-12-01"))

print(f"{len(urls)} ruedas de prensa")
print("Primera:", next(iter(urls.items())))
print("Última: ", list(urls.items())[-1])

55 ruedas de prensa
Primera: ('2019-12-12', 'https://www.ecb.europa.eu/press/press_conference/monetary-policy-statement/2019/html/ecb.is191212~c9e1a6ab3e.en.html')
Última:  ('2026-09-10', 'https://www.ecb.europa.eu/press/press_conference/monetary-policy-statement/2026/html/ecb.is260910~6a45359cfc.en.html')


In [3]:
# Control: número de ruedas por año
pd.Series(list(urls)).str[:4].value_counts().sort_index()

2019    1
2020    8
2021    8
2022    8
2023    8
2024    8
2025    8
2026    6
Name: count, dtype: int64

In [4]:
# Guardamos el listado para no tener que volver a consultarlo en los pasos siguientes
listado = pd.DataFrame({"date": list(urls), "url": list(urls.values())})
listado.to_csv(HISTORY_DIR / "press_conferences.csv", index=False)
print(f"Guardado: {HISTORY_DIR / 'press_conferences.csv'} ({len(listado)} filas)")

Guardado: /Users/eduardogonzalezarroyo/Developer/hawk-and-dove_BCE/data/history/press_conferences.csv (55 filas)


## 3. Descarga de las transcripciones

Descargamos el HTML de cada rueda de prensa y lo guardamos en `data/raw/ecb_html/` (fuera de git). Trabajar sobre una copia local tiene dos ventajas: el procesado es reproducible aunque la web del BCE cambie, y podemos ajustar el lector tantas veces como haga falta sin volver a descargar nada. Entre peticiones hacemos una pausa de un segundo para no sobrecargar el servidor.

In [5]:
# Descarga con caché: los ficheros que ya existen no se vuelven a pedir
listado = pd.read_csv(HISTORY_DIR / "press_conferences.csv")
html_dir = RAW_DIR / "ecb_html"
rutas = descargar_html(listado, html_dir)

tamanos_kb = pd.Series({r.stem: r.stat().st_size / 1024 for r in rutas})
print(f"{len(rutas)} ficheros en {html_dir}")
print(f"Tamaño total: {tamanos_kb.sum() / 1024:.1f} MB")

55 ficheros en /Users/eduardogonzalezarroyo/Developer/hawk-and-dove_BCE/data/raw/ecb_html
Tamaño total: 7.5 MB


In [6]:
# Control: los ficheros más pequeños. Una página de error pesaría muy poco
# comparada con una transcripción completa
tamanos_kb.sort_values().head(5).round(1)

2026-02-05    129.1
2026-03-19    130.1
2023-12-14    131.4
2025-07-24    132.6
2024-04-11    132.9
dtype: float64

## 4. Lectura de las transcripciones

Antes de fijar las reglas inspeccionamos el HTML de las 55 ruedas. Todas comparten estructura: el texto está en `<main> > div.section`, la declaración termina en un párrafo separador `* * *` y a partir de ahí empieza el turno de preguntas. Reglas aplicadas:

- **Preguntas.** Párrafos en negrita (al menos el 90 % del texto). Los párrafos en negrita consecutivos son la misma intervención del periodista (mismo `qa_id`), porque es habitual encadenar dos preguntas.
- **Hablante de cada respuesta.** Lo indica la etiqueta inicial, cuyo formato cambia según el año (`Lagarde:`, `President Lagarde:`, `President:`, `De Guindos:`, `Vice-President:`…). Sin etiqueta sigue hablando el mismo, y cada respuesta empieza atribuida a la presidenta. En las reuniones fuera de Fráncfort intervienen los gobernadores anfitriones (Stournaras en Atenas, 2023; Panetta en Florencia, 2025), que se marcan como `otro`.
- **Intervenciones fuera de pregunta (`remarks`).** Las palabras previas a la primera pregunta (por ejemplo, en la primera rueda de Lagarde) y el cierre tras un segundo separador (julio de 2026).
- **Moderador.** Una "pregunta" en negrita que nadie responde es el cierre del moderador (septiembre de 2022).
- **Limpieza.** Se eliminan las llamadas a notas al pie (`[1]`) y se conservan los ordinales (`3rd`).

**Limitación conocida:** si el vicepresidente responde sin etiqueta, su intervención se atribuye a la presidenta. Son casos aislados y no alteran la puntuación de postura, que agrega a todo el panel.

## 5. Segmentación en frases

Cada párrafo se divide en frases con **pySBD**, un segmentador basado en reglas pensado para los casos frecuentes en este corpus: abreviaturas ("Mr", "i.e.", iniciales como "J. Powell") y decimales ("2.6 per cent"). Frente a NLTK no requiere descargar modelos y falla menos con abreviaturas; frente a spaCy es más ligero para el mismo resultado.

La frase es la unidad de análisis porque es la granularidad del clasificador de postura (fase 2) y la que permite sincronizar texto, voz e imagen mediante marcas de tiempo.

In [7]:
# Construcción del corpus: lectura de las 55 ruedas y segmentación en frases (~15 s)
corpus = construir_corpus(listado, html_dir)

print(f"{len(corpus)} frases de {corpus['date'].nunique()} ruedas de prensa")
print("\nFrases por rol:")
print(corpus["role"].value_counts().to_string())
print("\nFrases por hablante:")
print(corpus["speaker"].value_counts().to_string())

15065 frases de 55 ruedas de prensa

Frases por rol:
role
answer       8344
statement    4006
question     2651
remarks        64

Frases por hablante:
speaker
presidenta        12112
periodista         2651
vicepresidente      221
otro                 74
moderador             7


In [8]:
# Control de la segmentación: longitud de las frases en palabras
n_palabras = corpus["text"].str.split().str.len()
print(n_palabras.describe().round(1).to_string())

print("\nFrases cortas más frecuentes (< 3 palabras):")
print(corpus.loc[n_palabras < 3, "text"].value_counts().head(8).to_string())

print(f"\nFrases de más de 80 palabras: {(n_palabras > 80).sum()}. Ejemplo:")
print(corpus.loc[n_palabras > 80, "text"].iloc[0])

count    15065.0
mean        22.4
std         14.0
min          1.0
25%         12.0
50%         20.0
75%         29.0
max        136.0

Frases cortas más frecuentes (< 3 palabras):
text
Thank you.         16
No.                10
Not quite.          3
Why?                3
Good afternoon.     3
That's one.         2
Yes.                2
It happens.         2

Frases de más de 80 palabras: 53. Ejemplo:
It will include aspects of inequality that are certainly rising in our economies, and all of those will be addressed with a view to exploring each and every corner of the business that we conduct as a central bank to see how those businesses are affected and how we can take them into account to better respond to the mission that we have, and to deliver on our mandate: serving the euro area citizens, and delivering on the mandate of price stability.


Las frases muy cortas son fórmulas habituales del turno de preguntas ("Thank you.", "No.") y las muy largas, frases encadenadas propias del lenguaje hablado; no son errores de segmentación. En la fase 2, las fórmulas sin contenido de política monetaria se marcarán como no relevantes.

## 6. Controles de calidad por rueda

Comprobamos que todas las ruedas tienen declaración, un número razonable de preguntas y que ninguna pregunta queda sin respuesta. Si algún control falla, la celda se detiene con un mensaje que indica cuál.

In [9]:
# Resumen por rueda de prensa
resumen = resumen_por_rueda(corpus)
resumen

,frases_declaracion,preguntas,frases_pregunta,frases_respuesta,frases_vicepresidente,frases_otros
date,,,,,,
2019-12-12,55,12,61,158,0,0
2020-01-23,54,12,73,205,0,0
2020-03-12,66,14,60,107,0,0
2020-04-30,67,9,38,166,0,0
2020-06-04,64,10,48,161,0,0
2020-07-16,67,9,39,172,0,0
2020-09-10,67,10,53,145,6,0
2020-10-29,65,12,50,186,0,0
2020-12-10,84,8,35,139,0,0


In [10]:
# Controles automáticos
preguntas = corpus[corpus["role"] == "question"].groupby("date")["qa_id"].apply(set)
respuestas = corpus[corpus["role"] == "answer"].groupby("date")["qa_id"].apply(set)
sin_respuesta = {d: preguntas[d] - respuestas.get(d, set()) for d in preguntas.index
                 if preguntas[d] - respuestas.get(d, set())}

assert corpus["date"].nunique() == 55, "Faltan ruedas de prensa"
assert (resumen["frases_declaracion"] > 0).all(), "Hay alguna rueda sin declaración"
assert (resumen["preguntas"] >= 5).all(), "Hay alguna rueda con menos de 5 preguntas"
assert not sin_respuesta, f"Preguntas sin respuesta: {sin_respuesta}"
print("Todos los controles superados")

Todos los controles superados


## 7. Guardado del corpus

El corpus se guarda en Parquet en `data/history/`, que sí se sube a git: la app y el chat lo necesitan nada más clonar el repositorio. Parquet conserva los tipos de cada columna (por ejemplo, `qa_id` como entero con vacíos) y ocupa una fracción de un CSV equivalente.

In [11]:
ruta_corpus = HISTORY_DIR / "transcripts.parquet"
corpus.to_parquet(ruta_corpus, index=False)
print(f"Guardado: {ruta_corpus} ({ruta_corpus.stat().st_size / 1024 / 1024:.2f} MB)")

Guardado: /Users/eduardogonzalezarroyo/Developer/hawk-and-dove_BCE/data/history/transcripts.parquet (0.99 MB)


## Resultado

El corpus contiene **15.065 frases de 55 ruedas de prensa** (diciembre de 2019 – septiembre de 2026):

- 4.006 frases de la declaración inicial.
- 2.651 frases de preguntas de periodistas, agrupadas en una media de 10 preguntas por rueda.
- 8.344 frases de respuestas: 8.049 de la presidenta, 221 del vicepresidente y 74 de gobernadores anfitriones.
- 64 frases de intervenciones fuera de pregunta (57 de la presidenta y 7 del moderador).

Todas las ruedas superan los controles de calidad. Este corpus alimenta la evaluación del clasificador de postura en la fase 2, el histórico de la fase 3 y el chat de la fase 4.